### Data Ingestion

In [1]:

import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from typing import Any, Dict, List

C:\Users\Aezion\AppData\Local\Temp\ipykernel_26972\3208276489.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: Revised Appointment Letter - Vikash Khede 2026 (004).pdf
  ✓ Loaded 7 pages

Processing: Saurabh mern 5+ (2).pdf
  ✓ Loaded 5 pages

Processing: Technical Assessment & Candidate Screening Playbook.pdf
  ✓ Loaded 7 pages

Processing: Welkin-Offer Letter - Saurabh M-03092026_signed.pdf
  ✓ Loaded 3 pages

Total documents loaded: 22


In [3]:
### Text splitting get into chunks
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size,length_function=len, chunk_overlap = chunk_overlap, separators=['\n\n','\n',' ',''])
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of chunk
    if split_docs:
        print('\n\n example of chunk')
        print(f"content:{split_docs[0].page_content[:200]}...")
        print(f'MetaData:{split_docs[0].metadata}')

    return split_docs


In [4]:
chunks = split_documents(all_pdf_documents)

split 22 documents into 47 chunks


 example of chunk
content:B-0816, 8th Floor, East Tower/ Tower B, Signature Towers, Brigade Golden, Triangle 
No. 50, Bidarahalli Hobli, Huskur Village, Bangalore, Karnataka -  560049 
Ph : 080-43755753 | www.aezion.com 
 
Let...
MetaData:{'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-05-07T11:58:20+05:30', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_enabled': 'true', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_setdate': '2024-01-31T03:56:46Z', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_method': 'Standard', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_name': 'defa4170-0d19-0005-0004-bc88714345d2', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_siteid': 'c2da5eb7-d0d5-4bf8-93da-ff6532c8d015', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_actionid': '0b71811d-fcce-42c8-83db-12106faa3e75', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_contentbits': '0', 

In [5]:
import chromadb
import numpy as np
import uuid
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer


c:\Users\Aezion\OneDrive - AEZION INC\Desktop\python\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class EmbeddingManager:
   """Handles document embedding generation using SentenceTransformer"""
   def __init__(self, model_name:str = 'all-MiniLM-L6-v2'):
      """ model_name: HuggingFace model name for sentence embeddings"""
      self.model_name = model_name
      self.model = None
      self._load_model()

   def _load_model(self):
        """Load the Sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded with {self.model.get_embedding_dimension()} dimension")
        except Exception as e:
            print(f"Error in loading model {self.model_name}: {e}")
            raise

   def generate_embedding(self, texts: list[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Genrating embadding for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3160.97it/s]


Model loaded with 384 dimension


### VectorStore

In [7]:
class VectorStore:
    def __init__(self, collection_name = "pdf_documents", presist_directory = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.presist_directory = presist_directory
        self.client = None
        self.collection = None
        self._initilise_store()

    def _initilise_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.presist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path = self.presist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self,documents: list[Any], embeddings: np.ndarray):
        if(len(documents) != len(embeddings)):
            raise ValueError('Number of documents and Number of embeddings must match')

        print(f"Adding {len(documents)} documents to vetor store...")

        ids = []
        metadatas = []
        documents_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4()}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embedding_list.append(embedding)

        try:
            self.collection.add(
                ids=ids,
                embeddings= embedding_list,
                documents=documents_text,
                metadatas= metadatas
            )
            print(f"Successfully added {len(embeddings)} embeddings to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [8]:
texts = [chunk.page_content for chunk in chunks]

## Generate Embeddings
embeddings = embedding_manager.generate_embedding(texts)

vector_store.add_documents(chunks, embeddings)

Genrating embadding for 47 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.01s/it]

Generated embeddings with shape: (47, 384)
Adding 47 documents to vetor store...
Successfully added 47 embeddings to vector store
Total documents in collection: 47


In [12]:
class RAGRetriever:
    def __init__(self,vector_store: VectorStore, embedding_manager:EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k:int = 5, score_threshold:float = 0.0)-> List[Dict[str, Any]]:
        print(f"Reterving docuemnt for query {query}")
        print(f"Top k:{top_k}, Score threshold:{score_threshold}")

        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        try:
            result = self.vector_store.collection.query(query_embeddings=[query_embedding.tolist()],n_results=top_k)
            retrieved_docs = []
            if result['documents'] and result['documents'][0]:
                ids = result['ids'][0]
                metadatas = result['metadatas'][0]
                documents = result['documents'][0]
                distances = result['distances'][0]

                for i,(id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                    ## Convert Squared L2 distance to exact Cosine Similarity
                    similarity_score = 1.0 - (distance / 2.0)

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':id,
                            'metadata':metadata,
                            'content': document,
                            'distance': distance,
                            'similarity_score': round(similarity_score, 4),
                            'rank': i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering) docs:{retrieved_docs}")
            else:
                print("No documents found")

        except Exception as e:
            print(f"Error in Retrival {e}")
        else:
            return retrieved_docs

rag_retriever = RAGRetriever(vector_store, embedding_manager)

rag_retriever.retrieve("vikas age")




Reterving docuemnt for query vikas age
Top k:5, Score threshold:0.0
Genrating embadding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 81.43it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering) docs:[{'id': 'doc_e00ec3e5-ed54-4208-8e38-7cc10d67e380_0', 'metadata': {'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_name': 'defa4170-0d19-0005-0004-bc88714345d2', 'moddate': '2026-05-11T14:45:49+05:30', 'author': 'user', 'producer': 'Microsoft® Word for Microsoft 365', 'page_label': '1', 'source_file': 'Revised Appointment Letter - Vikash Khede 2026 (004).pdf', 'content_length': 987, 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_actionid': '0b71811d-fcce-42c8-83db-12106faa3e75', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_enabled': 'true', 'file_type': 'pdf', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_siteid': 'c2da5eb7-d0d5-4bf8-93da-ff6532c8d015', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_method': 'Standard', 'doc_index': 0, 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_setdate': '2024-01-31T03:56:46Z', 'source': '..\\data\\pdf\\Revised Appointment Letter - Vikash Khede 2026

[{'id': 'doc_e00ec3e5-ed54-4208-8e38-7cc10d67e380_0',
  'metadata': {'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_name': 'defa4170-0d19-0005-0004-bc88714345d2',
   'moddate': '2026-05-11T14:45:49+05:30',
   'author': 'user',
   'producer': 'Microsoft® Word for Microsoft 365',
   'page_label': '1',
   'source_file': 'Revised Appointment Letter - Vikash Khede 2026 (004).pdf',
   'content_length': 987,
   'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_actionid': '0b71811d-fcce-42c8-83db-12106faa3e75',
   'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_enabled': 'true',
   'file_type': 'pdf',
   'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_siteid': 'c2da5eb7-d0d5-4bf8-93da-ff6532c8d015',
   'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_method': 'Standard',
   'doc_index': 0,
   'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_setdate': '2024-01-31T03:56:46Z',
   'source': '..\\data\\pdf\\Revised Appointment Letter - Vikash Khede 2026 (004).pdf',
   'creationdate': '2026-05-07

### RAG Pipeline- VectorDB to LLM output Generation

In [10]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

True

In [13]:
llm = ChatGroq(api_key=os.getenv('GROQ_API_KEY'), model="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)
def rag_simple(query, retriever: RAGRetriever, llm, top_k = 3):
    results = retriever.retrieve(query=query,top_k=top_k)
    context = "\n\n".join(result['content'] for result in results) if results else ""

    if not context:
        return "No relevent context found to answer the query"

    prompt = f"""Use the following context to answer the question
                Context: {context}
                Question: {query}
                Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content


print(rag_simple("age of vikas", retriever=rag_retriever,llm=llm))

Reterving docuemnt for query age of vikas
Top k:3, Score threshold:0.0
Genrating embadding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering) docs:[{'id': 'doc_e00ec3e5-ed54-4208-8e38-7cc10d67e380_0', 'metadata': {'page_label': '1', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_method': 'Standard', 'total_pages': 7, 'moddate': '2026-05-11T14:45:49+05:30', 'file_type': 'pdf', 'creator': 'Microsoft® Word for Microsoft 365', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_contentbits': '0', 'source_file': 'Revised Appointment Letter - Vikash Khede 2026 (004).pdf', 'page': 0, 'doc_index': 0, 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_enabled': 'true', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_siteid': 'c2da5eb7-d0d5-4bf8-93da-ff6532c8d015', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_setdate': '2024-01-31T03:56:46Z', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_actionid': '0b71811d-fcce-42c8-83db-12106faa3e75', 'content_length': 987, 'source': '..\\data\\pdf\\Revised Appointment Letter - Vikash Khede 2026 (004).pdf', 'prod

Vikash Khede was born on 05 May 1999. As of the letter’s date (1 January 2026), he is **26 years old** (he will turn 27 on 5 May 2026).


### Enhanced RAG Pipeline Features

In [26]:
def rag_advanced(query:str, llm, rag_retriever:RAGRetriever, temprature:float=0.0, top_k:int =3, min_score:float = 0.0, return_content:bool = True):
    results = rag_retriever.retrieve(query=query,score_threshold=min_score,top_k=top_k)
    if not results:
        return {'answer': 'No relevant context found', 'source':[], 'confidence':0.0, 'context':''}

    context = "\n\n".join([result['content'] for result in results])
    confidence = max([result['similarity_score'] for result in results])
    source = [{
        'source': result['metadata'].get('source_file', result['metadata'].get('source', 'unknown')),
        'page': result['metadata'].get('page', 'unknown'),
        'score': result['similarity_score'],
        'preview': result['content'][:300] + '...'
    } for result in results]

    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    output = {
        'answer' : response.content,
        'confidence': confidence,
        'source': source

    }
    if return_content:
        output['context'] = context

    return output



In [32]:
print(rag_advanced(query='whose CTC is 2200000', llm=llm, rag_retriever=rag_retriever))

Reterving docuemnt for query whose CTC is 2200000
Top k:3, Score threshold:0.0
Genrating embadding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering) docs:[{'id': 'doc_905619d2-12d1-4611-b1e1-093a95f5ce15_26', 'metadata': {'content_length': 987, 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_enabled': 'true', 'total_pages': 7, 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_setdate': '2024-01-31T03:56:46Z', 'file_type': 'pdf', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_actionid': '0b71811d-fcce-42c8-83db-12106faa3e75', 'page': 6, 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_method': 'Standard', 'author': 'user', 'moddate': '2026-05-11T14:45:49+05:30', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_siteid': 'c2da5eb7-d0d5-4bf8-93da-ff6532c8d015', 'source_file': 'Revised Appointment Letter - Vikash Khede 2026 (004).pdf', 'msip_label_1cc54b64-d55f-4688-9126-cbcd53906c8d_contentbits': '0', 'source': '..\\data\\pdf\\Revised Appointment Letter - Vikash Khede 2026 (004).pdf', 'creator': 'Microsoft® Word for Microsoft 365', 'page_label': '7', 'm

{'answer': 'Welkin\u202fSoft\u202fTech\u202fPvt\u202fLtd. (the employee’s CTC is ₹22,00,000).', 'confidence': 0.2996, 'source': [{'source': 'Revised Appointment Letter - Vikash Khede 2026 (004).pdf', 'page': 6, 'score': 0.2996, 'preview': 'B-0816, 8th Floor, East Tower/ Tower B, Signature Towers, Brigade Golden, Triangle \nNo. 50, Bidarahalli Hobli, Huskur Village, Bangalore, Karnataka -  560049 \nPh : 080-43755753 | www.aezion.com \n \n \n \nAnnexure A \nAEZION TECHNOLOGIES PRIVATE LIMITED \nEMPLOYEE COMPENSATION STRUCTURE \n   Vikash Khede  ...'}, {'source': 'Welkin-Offer Letter - Saurabh M-03092026_signed.pdf', 'page': 2, 'score': 0.2294, 'preview': 'WELKIN SOFT TECH PVT LTD \nCIN: U72200KA2008PTC047960, GSTIN: 29AAACW8227A1ZF \n \n #3484, 6th Floor, BDA 4th H Block, Raghuvanahalli, Banashankari 6th Stage, Bangalore – 560062 \nPh: 8050072642 | mail: hr@welkinsofttech.com | web: www.welkinsofttech.com \nSALARY BREAK UP  \n \nSalary Component Per Annum (...'}, {'source': 'Welkin-Offer